# CPT — Model Training on Google Colab

Trains all 4 learnable models (XGBoost, LightGBM, LSTM, TFT) for **SOL** and **DOGE**.
TimesFM is zero-shot and never trained.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or better)
2. Run all cells top-to-bottom — do not skip any
3. After the final save cell, download weights from Google Drive → `CPT_models/`
4. Place all downloaded files into your local `models/` directory

**Expected total time on T4:** ~45–60 min for both coins.

---
## 1 — Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO_URL = "https://github.com/its-dreOwO/CPT.git"
REPO_DIR = "/content/CPT"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest...")
    !git -C {REPO_DIR} pull

# Add project root to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Change kernel cwd so relative imports work
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("models/ will be at:", os.path.join(REPO_DIR, "models"))

---
## 2 — Install Dependencies

Colab already has `torch`, `pandas`, `numpy`, `xgboost`, `lightgbm`. Installing only what's missing.

In [ ]:
!pip install pytorch-forecasting lightning pydantic-settings structlog yfinance sqlalchemy tenacity ccxt --quiet
print("\nDone. All dependencies installed.")

---
## 3 — Create `.env`

Training is price-only — no real API keys needed.

In [ ]:
env_content = """DATABASE_URL=sqlite:///data/cpt.db
MODEL_DIR=models
REDIS_URL=redis://localhost:6379/0
FRED_API_KEY=placeholder
DISCORD_WEBHOOK_URL=placeholder
TWILIO_ACCOUNT_SID=placeholder
TWILIO_AUTH_TOKEN=placeholder
TWILIO_WHATSAPP_FROM=placeholder
ZALO_OA_ACCESS_TOKEN=placeholder
ZALO_APP_ID=placeholder
ZALO_APP_SECRET=placeholder
ZALO_REFRESH_TOKEN=placeholder
"""
env_path = os.path.join(REPO_DIR, ".env")
with open(env_path, "w") as f:
    f.write(env_content)
print(".env written to", env_path)

---
## 4 — Setup Database & Backfill Prices

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/setup_db.py"],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(result.stdout or "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("setup_db.py failed")
print("Database initialised.")

In [ ]:
result = subprocess.run(
    [sys.executable, "scripts/backfill_prices.py", "--coins", "SOL", "DOGE", "--days", "730"],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(result.stdout or "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("backfill_prices.py failed")
print("Backfill complete.")

In [ ]:
# Verify the DB has data before training
from storage.database import get_session
from storage.models import PriceData

with get_session() as session:
    sol_count = session.query(PriceData).filter(PriceData.coin == "SOL").count()
    doge_count = session.query(PriceData).filter(PriceData.coin == "DOGE").count()

print(f"SOL candles in DB:  {sol_count:,}")
print(f"DOGE candles in DB: {doge_count:,}")

if sol_count < 1000 or doge_count < 1000:
    raise RuntimeError("Not enough price data — re-run the backfill cell above.")
print("\nData looks good. Ready to train.")

---
## 5 — Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU")

---
## 6 — Training Helper

Run this cell once — it defines `run_training()` and `check_models()` used by all training cells.

In [ ]:
import subprocess, sys, glob, os

REPO_DIR = "/content/CPT"                          # absolute project root
MODELS_ABS = os.path.join(REPO_DIR, "models")      # absolute models/ path
os.makedirs(MODELS_ABS, exist_ok=True)             # ensure it exists now


def run_training(model: str, coin: str, epochs: int = None):
    """Run training script with real-time output streaming."""
    cmd = [sys.executable, "-u", "scripts/train_models.py",
           "--model", model, "--coin", coin]
    if epochs is not None:
        cmd += ["--epochs", str(epochs)]

    print(f"\n>>> {model.upper()} {coin}  (cwd={REPO_DIR})")
    print("-" * 60)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=REPO_DIR,           # ← always run from project root
        env={**os.environ,
             "PYTHONPATH": REPO_DIR,
             "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()

    if proc.returncode != 0:
        raise RuntimeError(
            f"{model} {coin} training failed (exit {proc.returncode}) "
            "— scroll up to see the error."
        )
    print("-" * 60)


def check_models(coin: str, expected_count: int):
    """List model files for the coin and warn if fewer than expected."""
    pattern = os.path.join(MODELS_ABS, f"*{coin.lower()}*")
    files = [f for f in glob.glob(pattern) if not f.endswith(".gitkeep")]
    print(f"\nModel files for {coin}: {len(files)} / {expected_count} expected")
    print(f"(looking in {MODELS_ABS})")
    for f in sorted(files):
        print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")
    if len(files) < expected_count:
        print("  WARNING: fewer files than expected.")
    else:
        print("  All expected files present.")


print(f"REPO_DIR   : {REPO_DIR}")
print(f"MODELS_ABS : {MODELS_ABS}")
print(f"models/ exists: {os.path.exists(MODELS_ABS)}")
print("Helper functions ready.")

---
## 7 — Train SOL Models

In [ ]:
run_training("xgboost", "SOL")

In [ ]:
run_training("lightgbm", "SOL")

In [ ]:
run_training("lstm", "SOL", epochs=100)

In [ ]:
run_training("tft", "SOL", epochs=50)

In [ ]:
# XGB: 3 files, LGBM: 3 files, LSTM: 1 file, TFT: 1 file = 8 total
check_models("sol", expected_count=8)

---
## 8 — Train DOGE Models

In [ ]:
run_training("xgboost", "DOGE")

In [ ]:
run_training("lightgbm", "DOGE")

In [ ]:
run_training("lstm", "DOGE", epochs=100)

In [ ]:
run_training("tft", "DOGE", epochs=50)

In [ ]:
check_models("doge", expected_count=8)

---
## 9 — Evaluate All Models

Target: **Directional Accuracy > 55%** and **Sharpe Ratio > 1.0**

In [ ]:
result = subprocess.run(
    [sys.executable, "-u", "scripts/evaluate_models.py", "--coin", "SOL", "--lookback", "90"],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
result = subprocess.run(
    [sys.executable, "-u", "scripts/evaluate_models.py", "--coin", "DOGE", "--lookback", "90"],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

---
## 10 — Save All Weights to Google Drive

In [ ]:
import shutil, glob, os

DRIVE_SAVE_DIR = "/content/drive/MyDrive/CPT_models"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

model_files = [f for f in glob.glob(os.path.join(MODELS_ABS, "*"))
               if not f.endswith(".gitkeep")]

if not model_files:
    raise RuntimeError(
        f"No model files found in {MODELS_ABS} — training may have failed above."
    )

for f in sorted(model_files):
    dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(f))
    shutil.copy(f, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"  Saved: {os.path.basename(f)}  ({size_mb:.1f} MB)")

print(f"\nAll {len(model_files)} file(s) saved to Google Drive > CPT_models/")

---
## 11 — Download & Import Locally

1. Open [Google Drive](https://drive.google.com) → find **`CPT_models/`** folder
2. Select all files → right-click → **Download** (saves as a zip)
3. Extract the zip
4. Copy **all files** into your local `models/` directory (replace any old weights)
5. Confirm locally:
   ```bash
   python scripts/evaluate_models.py --coin SOL --lookback 90
   python scripts/evaluate_models.py --coin DOGE --lookback 90
   ```